2.1 理论计算题
已知条件

输入图像大小：3 × 32 × 32（通道 × 高 × 宽）

卷积核数量：16 个

每个卷积核大小：3 × 5 × 5

填充（Padding）：2

步幅（Stride）：2

1. 输出特征图的尺寸（通道数 × 高 × 宽）
输出高度计算公式：
(32 + 2×2 - 5) / 2 = (32 + 4 - 5) / 2 = 31 / 2 = 15.5，向下取整再加 1
15 + 1 = 16

输出宽度计算相同，也是 16。
输出通道数等于卷积核数量，即 16。

答案： 16 × 16 × 16

2. 单个输出通道的一个像素值需要多少次乘法
每个卷积核的参数个数（不考虑偏置）：
3 × 5 × 5 = 75

每次计算一个输出像素时，需要卷积核的 75 个参数与输入对应位置的 75 个值分别相乘，然后求和。

答案： 75 次

In [3]:
import numpy as np

def max_pool2d(input, kernel_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    
    参数:
        input: 输入张量，形状为 (N, C, H, W) 或 (C, H, W) 或 (H, W)
        kernel_size: 池化窗口大小，int 或 (kh, kw)
        stride: 步幅，int 或 (sh, sw)，默认等于 kernel_size
        padding: 填充大小，int 或 (ph, pw)，默认 0
    
    返回:
        output: 池化后的输出张量
    """
    # 记录原始维度
    original_ndim = input.ndim
    
    # 处理输入维度，统一为 (N, C, H, W)
    if input.ndim == 2:
        input = input[np.newaxis, np.newaxis, :, :]  # (1, 1, H, W)
    elif input.ndim == 3:
        input = input[np.newaxis, :, :, :]  # (1, C, H, W)
    
    N, C, H, W = input.shape
    
    # 处理 kernel_size
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    # 处理 stride
    if stride is None:
        sh, sw = kh, kw
    elif isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    # 处理 padding
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    # 计算输出尺寸
    H_out = (H + 2 * ph - kh) // sh + 1
    W_out = (W + 2 * pw - kw) // sw + 1
    
    # 对输入进行填充
    padded = np.pad(input, ((0, 0), (0, 0), (ph, ph), (pw, pw)), 
                    mode='constant', constant_values=-np.inf)
    
    # 初始化输出
    output = np.zeros((N, C, H_out, W_out))
    
    # 滑动窗口进行池化
    for i in range(H_out):
        h_start = i * sh
        h_end = h_start + kh
        for j in range(W_out):
            w_start = j * sw
            w_end = w_start + kw
            window = padded[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2, 3))
    
    # 根据原始输入维度恢复输出形状
    if original_ndim == 2:
        # 输入是 (H, W)，输出应该是 (H_out, W_out)
        return output[0, 0, :, :]
    elif original_ndim == 3:
        # 输入是 (C, H, W)，输出应该是 (C, H_out, W_out)
        return output[0, :, :, :]
    else:
        # 输入是 (N, C, H, W)，输出保持 (N, C, H_out, W_out)
        return output


# ========== 测试代码 ==========
if __name__ == "__main__":
    # 测试 1: 基本功能，stride 默认等于 kernel_size
    x = np.random.randn(1, 1, 4, 4)
    result = max_pool2d(x, kernel_size=2, stride=2)
    print("测试 1 - 输出形状:", result.shape)  # 预期: (1, 1, 2, 2)
    
    # 测试 2: stride 小于 kernel_size（重叠池化）
    x = np.random.randn(1, 1, 5, 5)
    result = max_pool2d(x, kernel_size=3, stride=1, padding=0)
    print("测试 2 - 输出形状:", result.shape)  # 预期: (1, 1, 3, 3)
    
    # 测试 3: 带 padding
    x = np.random.randn(1, 1, 5, 5)
    result = max_pool2d(x, kernel_size=3, stride=2, padding=1)
    print("测试 3 - 输出形状:", result.shape)  # 预期: (1, 1, 3, 3)
    
    # 测试 4: 多通道输入
    x = np.random.randn(1, 3, 32, 32)
    result = max_pool2d(x, kernel_size=2, stride=2)
    print("测试 4 - 输出形状:", result.shape)  # 预期: (1, 3, 16, 16)
    
    # 测试 5: 批量输入
    x = np.random.randn(8, 16, 64, 64)
    result = max_pool2d(x, kernel_size=3, stride=2, padding=1)
    print("测试 5 - 输出形状:", result.shape)  # 预期: (8, 16, 32, 32)
    
    # 测试 6: 2D 输入
    x = np.random.randn(8, 8)
    result = max_pool2d(x, kernel_size=2, stride=2)
    print("测试 6 - 输出形状:", result.shape)  # 预期: (4, 4)
    
    # 测试 7: 3D 输入 (C, H, W)
    x = np.random.randn(3, 10, 10)
    result = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("测试 7 - 输出形状:", result.shape)  # 预期: (3, 5, 5)
    
# 测试 8: 修正版
try:
    import torch
    import torch.nn as nn
    
    np.random.seed(42)
    
    # 测试 4D 输入
    x_np = np.random.randn(2, 3, 10, 10)
    x_torch = torch.from_numpy(x_np).float()
    
    our_result = max_pool2d(x_np, kernel_size=3, stride=2, padding=1)
    torch_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    torch_result = torch_pool(x_torch).numpy()
    
    print("测试 8 - 4D 输入与 PyTorch 对比:", np.allclose(our_result, torch_result, atol=1e-6))
    
    # 测试 2D 输入 - 使用相同的池化参数
    x_np_2d = np.random.randn(8, 8)
    x_torch_2d = torch.from_numpy(x_np_2d).float().unsqueeze(0).unsqueeze(0)
    
    # 为 2D 输入创建新的池化层，参数与手动实现一致
    torch_pool_2d = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
    our_result_2d = max_pool2d(x_np_2d, kernel_size=2, stride=2, padding=0)
    torch_result_2d = torch_pool_2d(x_torch_2d).squeeze().numpy()
    
    print("测试 8 - 2D 输入与 PyTorch 对比:", np.allclose(our_result_2d, torch_result_2d, atol=1e-6))
    
except ImportError:
    print("测试 8 - 跳过（未安装 PyTorch）")

测试 1 - 输出形状: (1, 1, 2, 2)
测试 2 - 输出形状: (1, 1, 3, 3)
测试 3 - 输出形状: (1, 1, 3, 3)
测试 4 - 输出形状: (1, 3, 16, 16)
测试 5 - 输出形状: (8, 16, 32, 32)
测试 6 - 输出形状: (4, 4)
测试 7 - 输出形状: (3, 5, 5)
测试 8 - 4D 输入与 PyTorch 对比: True
测试 8 - 2D 输入与 PyTorch 对比: True


3.1 理论计算题
已知条件：

输入特征图通道数：C

输出特征图通道数：C

卷积层均不带偏置

1. 一个 5x5 卷积层的参数量
参数量 = 输入通道数 × 输出通道数 × 卷积核高 × 卷积核宽
= C × C × 5 × 5 = 25 × C²

答案：25 * C²

2. 两个串联的 3x3 卷积层的总参数量
每个 3x3 卷积层参数量：
C × C × 3 × 3 = 9 × C²

两个串联的总参数量：
9 × C² + 9 × C² = 18 × C²

答案：18 * C²

In [4]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块（Network in Network Block）
    
    结构：
        1. 普通卷积层 (kernel_size, stride, padding) + ReLU
        2. 1x1 卷积层 + ReLU
        3. 1x1 卷积层 + ReLU
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 第一个卷积层的卷积核大小 (int 或 tuple)
        stride: 第一个卷积层的步幅 (int 或 tuple，默认 1)
        padding: 第一个卷积层的填充 (int 或 tuple，默认 0)
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        
        self.block = nn.Sequential(
            # 第一个卷积层 (普通卷积)
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(inplace=True),
            
            # 第二个卷积层 (1x1 卷积)
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            
            # 第三个卷积层 (1x1 卷积)
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)


# ========== 使用示例 ==========
if __name__ == "__main__":
    # 创建一个 NiN 块实例
    # 例如：输入 3 通道，输出 96 通道，卷积核 3x3，步幅 1，填充 1
    nin_block = NiNBlock(in_channels=3, out_channels=96, kernel_size=3, stride=1, padding=1)
    
    # 测试输入：batch_size=1, channels=3, height=32, width=32
    x = torch.randn(1, 3, 32, 32)
    
    # 前向传播
    output = nin_block(x)
    
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")
    print(f"NiN 块结构:\n{nin_block}")

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 96, 32, 32])
NiN 块结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU(inplace=True)
  )
)


4.1 理论计算题
已知：

小批量样本特征值：x1 = 2，x2 = 4，x3 = 6，x4 = 8

缩放参数 γ = 2

平移参数 β = 1

ε = 0（用于避免除零，这里为 0 说明除数可能为零，但数据非零方差没问题）

第 1 步：计算批量均值 μ
μ = (x1 + x2 + x3 + x4) / 4
= (2 + 4 + 6 + 8) / 4
= 20 / 4 = 5

第 2 步：计算批量方差 σ²（总体方差，分母用 m）
σ² = [(x1-μ)² + (x2-μ)² + (x3-μ)² + (x4-μ)²] / 4

(2-5)² = 9

(4-5)² = 1

(6-5)² = 1

(8-5)² = 9

σ² = (9 + 1 + 1 + 9) / 4 = 20 / 4 = 5

σ = √5 ≈ 2.236（本题只需保留 σ² 形式）

第 3 步：标准化（减去均值除以标准差）
公式：(xi - μ) / √(σ² + ε)

由于 ε = 0，标准差 = √5

z1 = (2 - 5) / √5 = -3 / √5

z2 = (4 - 5) / √5 = -1 / √5

z3 = (6 - 5) / √5 = 1 / √5

z4 = (8 - 5) / √5 = 3 / √5

第 4 步：缩放和平移（γ × z + β）
γ = 2，β = 1

y1 = 2 × (-3 / √5) + 1 = 1 - (6 / √5)

y2 = 2 × (-1 / √5) + 1 = 1 - (2 / √5)

y3 = 2 × (1 / √5) + 1 = 1 + (2 / √5)

y4 = 2 × (3 / √5) + 1 = 1 + (6 / √5)

最终答案（保留 √5 形式）
y1 = 1 - 6 / √5
y2 = 1 - 2 / √5
y3 = 1 + 2 / √5
y4 = 1 + 6 / √5



In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residual(nn.Module):
    """
    残差块（Residual Block）
    
    结构：
        - 两个 3x3 卷积层，每个卷积层后跟一个批量归一化层和 ReLU 激活
        - 如果 use_1x1conv=True，则使用 1x1 卷积调整输入残差路径
        - 输出 = 主路径输出 + 残差路径输入
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        stride: 第一个卷积层的步幅（默认为 1）
        use_1x1conv: 是否使用 1x1 卷积调整输入形状（默认为 False）
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        
        # 第一个卷积层：3x3，步幅为 stride，填充为 1（保持分辨率）
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层：3x3，步幅为 1，填充为 1
        self.conv2 = nn.Conv2d(out_channels, out_channels, 
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 残差路径：如果 use_1x1conv=True，使用 1x1 卷积调整输入
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 
                                      kernel_size=1, stride=stride, bias=False)
            self.bn_shortcut = nn.BatchNorm2d(out_channels)
        else:
            self.shortcut = None
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # 主路径
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 残差路径（恒等映射或 1x1 卷积）
        if self.shortcut is not None:
            identity = self.shortcut(x)
            identity = self.bn_shortcut(identity)
        else:
            identity = x
        
        # 相加后激活
        out += identity
        out = self.relu(out)
        
        return out


# ========== 使用示例 ==========
if __name__ == "__main__":
    # 示例 1：输入输出通道相同，不使用 1x1 卷积
    block1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
    x1 = torch.randn(1, 64, 32, 32)
    out1 = block1(x1)
    print(f"示例1 - 输入: {x1.shape} -> 输出: {out1.shape}")
    
    # 示例 2：输入输出通道不同，使用 1x1 卷积调整
    block2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
    x2 = torch.randn(1, 64, 32, 32)
    out2 = block2(x2)
    print(f"示例2 - 输入: {x2.shape} -> 输出: {out2.shape}")
    
    # 打印模型结构
    print(f"\n残差块结构 (use_1x1conv=True):\n{block2}")

示例1 - 输入: torch.Size([1, 64, 32, 32]) -> 输出: torch.Size([1, 64, 32, 32])
示例2 - 输入: torch.Size([1, 64, 32, 32]) -> 输出: torch.Size([1, 128, 16, 16])

残差块结构 (use_1x1conv=True):
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (shortcut): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
  (bn_shortcut): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)


5.1 理论计算题
1. 为什么底层学习率较小，顶层学习率较大？
原因主要有以下几点：

底层特征具有通用性：深度网络的底层（靠近输入层）通常学习到边缘、颜色、纹理等通用低级特征。这些特征在不同任务和数据集之间具有较好的可迁移性，因此不需要大幅调整，甚至可以直接复用。

顶层特征具有任务特异性：顶层（靠近输出层）学习到的高级特征与具体任务类别高度相关。当切换到一个新的目标数据集时，原来的分类器（顶层）不再适用，需要重新训练或大幅调整。

防止破坏预训练知识：如果对底层使用过大的学习率，可能会破坏预训练模型中已经学到的通用特征，导致迁移效果变差，甚至比不上从头训练。

加速收敛与防止过拟合：底层使用小学习率（或冻结）可以减少需要优化的参数量，使模型更快收敛，同时降低在小数据集上过拟合的风险。

结论：

底层特征提取层 → 学习率小（或冻结）

顶层输出层 → 学习率大（重新训练）

2. 目标数据集非常小且与源数据集非常相似时的微调策略
为了防止过拟合，可以采取以下策略：

冻结除最后几层外的所有层：只微调最终的分类层（全连接层），将前面所有特征提取层的参数全部冻结。这样可以最大限度地保留源数据集学到的通用特征，避免小数据集导致过拟合。

使用更小的学习率：即使对于要微调的层，也应使用较小的学习率（比如预训练学习率的 1/10 或更小），避免模型参数发生剧烈变化。

更强的正则化：增加 Dropout 比例、L2 正则化（权重衰减）系数，或使用早停法（Early Stopping）。

数据增广：对目标数据集应用丰富的数据增强（随机裁剪、翻转、旋转、颜色抖动等），以扩充有效训练样本数量。

使用更少的训练轮数：因为数据集很小且与源数据相似，模型很容易快速过拟合，因此训练轮数不宜过多。

使用标签平滑（Label Smoothing）：可以防止模型对训练样本过度自信，从而提升泛化能力。

典型策略总结（最常见做法）：

冻结骨干网络（Backbone），只重新训练最后一层（或最后几层），并使用较小的学习率。

In [6]:
import torchvision.transforms as transforms
from PIL import Image
import torch

# ========== 创建图像增广管道 ==========
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放到 224x224
    #    scale: 裁剪面积比例范围 [0.08, 1.0]
    #    ratio: 裁剪宽高比范围，默认 [3/4, 4/3]，这里保持默认
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度、饱和度
    #    ColorJitter 的参数分别对应：亮度、对比度、饱和度、色调（这里只用到前三个）
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor(),
])

# ========== 使用示例 ==========
if __name__ == "__main__":
    # 示例：加载一张图片并应用增广
    # 这里用随机图像代替，实际使用时替换为真实图片路径
    # img = Image.open("your_image.jpg")
    
    # 创建一张随机测试图像（RGB，300x300）
    test_img = Image.new('RGB', (300, 300), color=(128, 128, 128))
    
    # 应用增广管道
    augmented_tensor = augmentation_pipeline(test_img)
    
    print(f"增广后的张量形状: {augmented_tensor.shape}")  # torch.Size([3, 224, 224])
    print(f"张量值范围: [{augmented_tensor.min():.3f}, {augmented_tensor.max():.3f}]")
    
    # 打印完整的增广管道结构
    print(f"\n增广管道:\n{augmentation_pipeline}")


# ========== 另一种写法：分步理解 ==========
"""
每个 transform 的作用说明：

1. RandomResizedCrop(size=224, scale=(0.08, 1.0))
   - 随机裁剪图像的一个区域，裁剪面积占原图面积的比例在 0.08~1.0 之间
   - 然后将裁剪区域缩放到 224x224 像素

2. RandomHorizontalFlip(p=0.5)
   - 以 50% 的概率随机水平翻转图像

3. ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)
   - brightness=0.5: 亮度调整因子在 [1-0.5, 1+0.5] = [0.5, 1.5] 范围内随机
   - contrast=0.5: 对比度调整因子在 [0.5, 1.5] 范围内随机
   - saturation=0.5: 饱和度调整因子在 [0.5, 1.5] 范围内随机
   - 也可以单独指定每个参数的随机范围，如 brightness=(0.8, 1.2)

4. ToTensor()
   - 将 PIL Image 或 numpy.ndarray (H x W x C) 转换为 PyTorch Tensor
   - 自动将像素值从 [0, 255] 缩放到 [0.0, 1.0]
   - 同时将通道维度从最后一维移到第一维 (C x H x W)
"""

增广后的张量形状: torch.Size([3, 224, 224])
张量值范围: [0.667, 0.667]

增广管道:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


'\n每个 transform 的作用说明：\n\n1. RandomResizedCrop(size=224, scale=(0.08, 1.0))\n   - 随机裁剪图像的一个区域，裁剪面积占原图面积的比例在 0.08~1.0 之间\n   - 然后将裁剪区域缩放到 224x224 像素\n\n2. RandomHorizontalFlip(p=0.5)\n   - 以 50% 的概率随机水平翻转图像\n\n3. ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)\n   - brightness=0.5: 亮度调整因子在 [1-0.5, 1+0.5] = [0.5, 1.5] 范围内随机\n   - contrast=0.5: 对比度调整因子在 [0.5, 1.5] 范围内随机\n   - saturation=0.5: 饱和度调整因子在 [0.5, 1.5] 范围内随机\n   - 也可以单独指定每个参数的随机范围，如 brightness=(0.8, 1.2)\n\n4. ToTensor()\n   - 将 PIL Image 或 numpy.ndarray (H x W x C) 转换为 PyTorch Tensor\n   - 自动将像素值从 [0, 255] 缩放到 [0.0, 1.0]\n   - 同时将通道维度从最后一维移到第一维 (C x H x W)\n'